In [ ]:
# 실습에 필요한 패키지 설치 (최초 1회 실행)
!pip install -q torch tokenizers


In [ ]:
# ========== 사전 준비: 2_1에서 만든 토크나이저 불러오기 ==========
# 이 노트북은 2_1_Tokenization에 이어서 진행한다.
#
# 2_1에서 tokenizer.save_model('.', 'nsmc')로 저장한 'nsmc-vocab.txt'를 불러온다.
# → 20만 문장을 다시 학습할 필요가 없어 몇 초 만에 준비가 끝난다.
# → 파일이 없다면(2_1을 건너뛴 경우) 그 자리에서 학습한다.
import os
from tokenizers import BertWordPieceTokenizer

VOCAB_FILE = 'nsmc-vocab.txt'

if os.path.exists(VOCAB_FILE):
    # 사전 파일을 주고 생성하면 [CLS]/[SEP] 후처리기까지 함께 설정된다 (2_1 마지막 참고)
    tokenizer = BertWordPieceTokenizer(VOCAB_FILE, lowercase=False, strip_accents=False)
    print(f'저장된 토크나이저를 불러왔습니다: {VOCAB_FILE}')
else:
    print('저장된 사전이 없어 새로 학습합니다. (1~2분 소요)')
    tokenizer = BertWordPieceTokenizer(lowercase=False, strip_accents=False)
    tokenizer.train(
        files='nsmc.txt',        # 2_1에서 만든 정제된 리뷰 데이터
        vocab_size=30000,        # 단어 사전 크기
        min_frequency=2,         # 최소 2번 이상 등장한 단어만 포함
        special_tokens=['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]'],
    )
    tokenizer.save_model('.', 'nsmc')

print(f'토크나이저 준비 완료 (사전 크기: {tokenizer.get_vocab_size()})')


# 2. 임베딩

- 토큰화를 통해 얻은 ID 시퀀스 정보에 의미를 부여하는 과정
- `45` 정수 ID는 `44` 정수 ID나 `46` 정수 ID와 아무런 수학적 관계가 없음.
- 따라서, 각 ID 값들을 의미 있는 공간의 **좌표.** 즉, 벡터로 변환 할 필요가 있음.

> **📌 잠깐 — 2_0에서 뽑은 "임베딩"과 지금 만들 "임베딩"은 다르다**
>
> 2_0에서는 `output.last_hidden_state`를 임베딩이라고 불렀다.
> 이 노트북에서 만들 것은 그것과 **다른 것**이므로 다시 한번 정리하고 시작하자.
>
> ```
> input_ids ──> [임베딩 층] ──> [Transformer 층 12개] ──> last_hidden_state
>                  ↑                                          ↑
>            지금 만들 것 (2_2)                       2_0에서 뽑은 것
>            단어당 벡터 1개 고정                   문장 문맥이 반영된 벡터
> ```
>
> | | 임베딩 **층** (지금) | `last_hidden_state` (2_0) |
> |---|---|---|
> | 정체 | 정수 ID를 벡터로 바꾸는 **표** | 모델을 다 통과한 **결과** |
> | 문맥 | 반영 안 됨 | 반영됨 |
> | 위치 | 모델의 입구 | 모델의 출구 |
>
> 👉 임베딩 층은 **모델의 첫 관문**이다. 문맥은 그 뒤의 층들이 만들어 준다.


## 2-1. 임베딩 종류와 발전 과정

- **어떻게 하면 단어의 의미를 벡터에 가장 잘 담을 수 있을까?**

### 2-1-0. 원핫 인코딩

- 가장 직관적이고 간단한 단어 표현 방식
- 단어 사전에 있는 단어의 수 만큼 벡터를 생성
- 표현 하고 싶은 단어의 인덱스 위치만 1로 표기, 나머지는 모두 0으로 처리
- 한계점
    1. **차원의 저주:** 단어 사전의 크기가 커지면 벡터도 그 만큼 커지지만, 대부분이 0인 낭비가 발생
    2. **의미 관계 표현의 부재**
        - 어떤 단어가 되었든, 단순히 단어 사전에 등장한 위치에만 의미가 있으므로 단어 간의 관계를 나타낼 수 가 없어짐.

In [ ]:
# ========== 원핫 인코딩 예시 ==========
# 단어 사전이 3개 단어라면, 각 단어를 3차원 벡터로 표현
vocab = ['apple', 'banana', 'orange']

apple  = [1, 0, 0]  # apple의 위치만 1
banana = [0, 1, 0]  # banana의 위치만 1
orange = [0, 0, 1]  # orange의 위치만 1

# 문제점 1: 단어 사전이 30,000개면? → 30,000차원 벡터, 대부분이 0 (메모리 낭비)
# 문제점 2: apple과 orange는 둘 다 과일인데, 벡터 간 유사도는 0이다.
#           → 단어 간의 의미 관계를 전혀 표현하지 못한다.

# 실제로 유사도를 계산해서 확인해 보자.
import torch
import torch.nn.functional as F

a = torch.tensor(apple, dtype=torch.float)
b = torch.tensor(banana, dtype=torch.float)
o = torch.tensor(orange, dtype=torch.float)

print(f'apple vs banana: {F.cosine_similarity(a, b, dim=0):.1f}')
print(f'apple vs orange: {F.cosine_similarity(a, o, dim=0):.1f}')
print('→ 전부 0. 어떤 단어 쌍을 골라도 항상 0이다.')
print('   "과일끼리는 좀 비슷하다"는 정보를 담을 방법이 아예 없다.')

# 이 한계를 극복하기 위해 등장한 것이 "임베딩"이다.


### 2-1-1. 정적 임베딩

- 단어의 의미를 저차원의 `실수 벡터`에 압축하는 패러다임
    - **Word2Vec:** 단어의 의미는 주변 단어에 의해 결정된다는 아이디어 기반
        1. **CBOW (Continuous Bag-of-Words)**
            - 주변 단어들로 중심 단어를 예측하는 방식
            - “사과의 ㅇㅇㅇ 빨갛다” → ㅇㅇㅇ에 올 수 있는 단어는?
        2. **Skip-gram**
            - 중심 단어로 주변 단어들을 예측하는 방식
            - “ㅁㅁㅁ 창문을 OOO” → ㅁㅁㅁ와 ㅇㅇㅇ에 올 수 있는 단어는?
    - **GloVe (Global Vectors for Word Representation)**
        - 말뭉치 전체의 통계 정보를 사용해, 전체적인 통계를 먼저 계산하고 이를 벡터화
        

### 2-1-2. RNN/LSTMs

- 문맥적 임베딩을 시도한 첫 주류 모델
- 문장이 길어지면 **장기 의존성 문제**(앞쪽 정보 소실)와 **느린 속도** 문제 발생

### 2-1-3. Transformer / Positional Embedding / BERT

1. **Transformer** 
    - RNN의 순차 처리 방식에서 벗어남 → 문장 전체를 한 번에 병렬 처리하는 구조
    - **셀프 어텐션(Self-Attention):** 문장 내 모든 단어 간의 관계 중요도를 한 번에 계산
2. **Positional Embedding**
    - 단어의 순서 정보를 모델에 알려주기 위한 장치
    - 단어의 위치마다 고유한 벡터를 생성
    - 원래의 단어 의미 임베딩에 더해 줌
3. **BERT** (Bidirectional Encoder Representations from Transformers)
    - 트랜스포머의 인코더 구조만 사용
    - 문장의 **양방향** 문맥을 동시에 학습

## 2-2. 임베딩 층(Embedding Layer)이란?

- pytorch의 `nn.Embedding` 모듈을 활용하여 변환 과정을 진행 할 것
    - 학습 가능한 거대한 Lookup Table. (**가중치 행렬)**
- 행렬 구성: (단어 사전의 크기, 임베딩 벡터의 차원)
    - 행: 단어 사전에 있는 **토큰 하나하나**에 해당
    - 열: 벡터의 차원을 나타냄. 차원의 크기가 클 수록 더 복잡한 관계를 벡터에 담을 수 있음.
    - ex) `45`번 ID의 토큰 `나는` 을 10차원 벡터화 한다면,
    이 가중치 행렬의 45번 행에 `[0.21, -1.35, 0.07, 0.88, -0.42, 1.10, -0.63, 0.05, 0.31, -0.19]` 형태의 벡터가 저장되는 셈
        - 📌 임베딩 벡터는 **정수가 아니라 실수(소수)** 값이다. 조금씩 미세하게 조정되며 학습되어야 하기 때문이다.

> **💡 왜 하필 768차원일까?**
>
> 정답은 없다. 다만 관례가 있다.
> - **너무 작으면**: 담을 수 있는 의미가 적어 서로 다른 단어가 비슷한 벡터가 되어 버린다.
> - **너무 크면**: 메모리를 많이 쓰고, 데이터가 부족하면 과적합된다.
> - 실무에서는 보통 **64의 배수**를 쓴다. GPU가 데이터를 64개 단위로 묶어 처리할 때 가장 효율적이기 때문이다.
> - `768`은 BERT-base가 쓰는 값이라 사실상 표준처럼 자리 잡았다. (768 = 64 × 12)
>
> ⚠️ 참고로 768은 흔히 말하는 "2의 제곱수"가 아니다. (2의 제곱수는 512, 1024, 2048)


In [ ]:
import torch
import torch.nn as nn

# ⭐ 하드코딩(30000) 대신 실제 사전 크기를 사용한다.
#    학습 결과가 목표치에 못 미칠 수 있으므로, 실제 값을 쓰는 것이 안전하다.
#    임베딩 표가 사전보다 작으면 IndexError가 발생한다.
vocab_size = tokenizer.get_vocab_size()   # 챕터 1에서 만든 단어 사전의 실제 크기
embedding_dim = 768                        # 각 단어를 표현할 벡터의 차원

# ========== 임베딩 층 생성 ==========
# nn.Embedding: 학습 가능한 거대한 Lookup Table (가중치 행렬)
# 크기: (vocab_size, 768) = 단어 수 × 768차원 벡터
#
# 원핫 인코딩과의 차이:
# - 원핫: (30000, 30000) → 단어당 30,000차원, 대부분 0
# - 임베딩: (30000, 768) → 단어당 768차원, 밀집(dense) 벡터
#
# 처음에는 임의의 값(랜덤)으로 초기화되어 있다.
# 학습 과정에서 단어 간의 의미 관계를 포착하도록 업데이트될 예정이다.
# (2_3에서 사전학습된 모델의 임베딩을 이식할 것)
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

print(f'임베딩 가중치 행렬 크기: {embedding_layer.weight.shape}')
print(f'→ 단어 {vocab_size:,}개, 각 {embedding_dim}차원 벡터')
print(f'→ 학습 파라미터 수: {vocab_size * embedding_dim:,}개')

# 예시: 45번 ID의 임베딩 벡터 확인 (현재는 무작위 값)
word_index = 45
print(f'\n45번 토큰: {tokenizer.id_to_token(word_index)}')
word_vector = embedding_layer.weight[word_index]
print(f'45번 토큰의 임베딩 벡터 (처음 5개 값): {word_vector[:5].tolist()}')
print('→ 현재는 무작위 값. 학습 후에는 의미 있는 좌표가 된다.')

# ========== "조회(lookup)"임을 직접 확인 ==========
# 임베딩 층에 ID를 넣은 결과와, 가중치 표에서 그 번호의 행을 꺼낸 것이 완전히 같다.
out = embedding_layer(torch.tensor([word_index]))
print(f'\n층에 넣은 결과와 표의 45번 행이 같은가? {torch.equal(out[0], word_vector)}')
print('→ 임베딩은 복잡한 계산이 아니라 "표에서 행 하나 꺼내오기"이다.')


## 2-3. 미리보기: 임베딩 벡터, 어디에 쓸까?

- Word2Vec 등의 학습 과정을 통해 **단어의 의미와 문맥적 관계**를 벡터로 표현한다면,
- 이 벡터들을 활용해, **단어 간의 유사성**을 측정할 수 있음
    - 현재는 무작위 벡터값으로 초기화 되어 있음에 유의

### 2-3-1. 코사인 유사도

- 두 벡터가 고차원 공간에서 얼마나 `같은 방향`을 향하고 있는지를 측정하는 지표
1. 두 벡터 사이의 각도의 코사인 값을 계산
    - 각도가 작을 수록 코사인 값은 1에 가까워 짐. (같은 방향을 바라본다.)
    - 각도가 클 수록 코사인 값은 -1에 가까워 짐. (완전히 반대 방향을 바라본다.)
2. 수식은… 지금은 스킵

In [ ]:
import torch.nn.functional as F

# ========== 코사인 유사도로 유사 단어 찾기 (교안 슬라이드 20 관련) ==========
# 코사인 유사도: 두 벡터가 같은 방향을 향하는지 측정 (-1 ~ 1)
# → 1에 가까울수록 의미가 비슷, -1에 가까울수록 반대 의미

target_word = '영화'
top_k = 5

# 기준 단어의 ID와 벡터 조회
target_id = tokenizer.token_to_id(target_word)

# ⚠️ 방어 코드: 사전에 없는 단어를 넣으면 token_to_id가 None을 반환하고,
#    아래에서 TypeError가 발생한다. 다른 단어로 실험할 때 자주 만나는 오류다.
if target_id is None:
    raise ValueError(f"'{target_word}'는 단어 사전에 없습니다. 다른 단어로 시도해 보세요.")

target_vector = embedding_layer.weight[target_id]

# 전체 단어 벡터와 코사인 유사도 계산
# torch.no_grad(): 학습이 아니라 단순 조회이므로 미분 정보를 만들지 않는다
with torch.no_grad():
    all_vectors = embedding_layer.weight
    similarities = F.cosine_similarity(
        target_vector.unsqueeze(0),  # [768] → [1, 768] (비교를 위해 차원 추가)
        all_vectors,                  # [vocab_size, 768]
        dim=1                         # 768차원 방향으로 유사도 계산
    )

    # 유사도가 높은 Top-K 단어 찾기 (자기 자신 제외)
    top_scores, top_indices = torch.topk(similarities, k=top_k + 1)

print(f"⚠️ [학습 전] '{target_word}'와 가장 유사한 단어 Top {top_k}:")
for i in range(1, top_k + 1):
    similar_word_id = top_indices[i].item()
    similar_word = tokenizer.id_to_token(similar_word_id)
    score = top_scores[i].item()
    print(f'  {i}순위: {similar_word} (유사도: {score:.4f})')

print('\n→ 현재는 무작위 벡터이므로 의미 없는 단어가 나온다.')
print(f'→ 유사도 값도 0 근처({top_scores[1].item():.3f})에 머문다.')
print('   768차원 공간에서 무작위 벡터 두 개는 거의 항상 직각(유사도 0)에 가깝기 때문이다.')
print('→ 2_3에서 사전학습 모델의 지식을 이식한 후 다시 확인하면 결과가 달라질 것!')
